# Phase 4 — 딥러닝: 오토인코더 기반 이상탐지

지금까지 성적표 (val macro-F1 기준):
- 마할라노비스 (Phase 2): **0.7324** ← 현재 1등
- Isolation Forest: 0.6912
- One-Class SVM: 0.6017
- LOF: 0.50~0.52

이 노트북에서 할 일:
1. 오토인코더(autoencoder) 개념 이해 — 신경망으로 데이터를 압축했다가 복원하는 구조
2. 신경망 학습에 맞게 데이터 스케일링
3. PyTorch로 오토인코더 직접 설계 및 학습
4. 학습 손실곡선 확인
5. val의 재구성 오차(reconstruction error)로 이상탐지 점수 계산, threshold 튜닝
6. Phase 2~3 비교표에 추가, 네 번째 제출 여부 결정

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

train = pd.read_csv('../data/train.csv')
val = pd.read_csv('../data/val.csv')
test = pd.read_csv('../data/test.csv')
feature_cols = [c for c in train.columns if c.startswith('V')]

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

y_true = val['Class'].values

print('torch:', torch.__version__, '/ CUDA:', torch.cuda.is_available())
print('train:', train.shape, '/ val:', val.shape, '/ test:', test.shape)

torch: 2.8.0+cpu / CUDA: False
train: (113842, 31) / val: (28462, 32) / test: (142503, 31)


## 공통 함수 재사용

Phase 2~3에서 검증까지 마친 macro-F1, threshold 스윕 로직을 그대로 가져옵니다.

In [2]:
def confusion_counts(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn

def precision_recall_f1(y_true, y_pred):
    tp, tn, fp, fn = confusion_counts(y_true, y_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def macro_f1(y_true, y_pred):
    _, _, f1_class1 = precision_recall_f1(y_true, y_pred)
    _, _, f1_class0 = precision_recall_f1(1 - y_true, 1 - y_pred)
    return (f1_class0 + f1_class1) / 2, f1_class0, f1_class1

def find_best_threshold(y_true, scores, max_k=2000):
    # 점수가 높은 순으로 상위 k건을 사기로 예측하는 모든 경우(k=1~max_k)를 빠짐없이 탐색합니다
    # (사기가 극소수라 의미 있는 threshold는 상위 몇십 건 안에 몰려 있어서, 균등 간격 탐색은 놓치기 쉽습니다)
    order = np.argsort(-scores)
    sorted_y = y_true[order]
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    k = np.arange(1, max_k + 1)
    tp = np.cumsum(sorted_y)[:max_k]   # 상위 k건 중 진짜 사기 수
    fp = k - tp
    fn = n_pos - tp
    tn = n_neg - fp
    # F1 = 2TP / (2TP + FP + FN): 사기 기준 F1과 정상 기준 F1을 평균내면 macro-F1
    macro = (2 * tp / (2 * tp + fp + fn) + 2 * tn / (2 * tn + fn + fp)) / 2
    best_idx = np.argmax(macro)
    thresholds = scores[order][:max_k]   # k번째로 높은 점수 = 'score >= threshold면 사기'로 예측할 때의 threshold
    return thresholds[best_idx], macro[best_idx], thresholds, macro

results = {
    '마할라노비스(Phase2)': {'val_macro_f1': 0.7324},
    'Isolation Forest': {'val_macro_f1': 0.6912},
    'One-Class SVM': {'val_macro_f1': 0.6017},
    'LOF(RobustScaler)': {'val_macro_f1': 0.5220},
}